# Análise Telemetria Sistema 1 — 06/06/2026

Análise dos sinais coletados durante incubação (21/05 → 06/06).
Objetivo: validar/falsificar tese de edge antes de implementar Adaptive Sizing.

In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Carregar signals.jsonl
signals_file = Path("../logs/signals.jsonl")
records = []
with open(signals_file, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
df = df.sort_values("timestamp_utc").reset_index(drop=True)

print(f"Total de sinais: {len(df)}")
print(f"Período: {df['timestamp_utc'].min()} → {df['timestamp_utc'].max()}")
print(f"Símbolos: {df['symbol'].unique()}")
print(f"Tipos de sinal: {df['signal'].value_counts().to_dict()}")

In [ ]:
# Expandir features em colunas
features_df = pd.json_normalize(df["features"])
df_full = pd.concat([df.drop("features", axis=1), features_df.add_prefix("feat_")], axis=1)

# Filtrar sinais com outcome de 1h preenchido
analyzed = df_full[df_full["outcome_1h"].notna() & (df_full["outcome_1h"] != "ERROR")].copy()
analyzed["outcome_1h"] = pd.to_numeric(analyzed["outcome_1h"], errors="coerce")
analyzed = analyzed.dropna(subset=["outcome_1h"])

print(f"Sinais analisáveis (com outcome_1h): {len(analyzed)}")
print(f"Sinais BUY: {(analyzed['signal']=='BUY').sum()}")
print(f"Sinais SELL: {(analyzed['signal']=='SELL').sum()}")

In [ ]:
# Win rate por tipo de sinal
def win_rate(group, signal_type):
    if signal_type == "BUY":
        wins = (group["outcome_1h"] > 0).sum()
    else:  # SELL
        wins = (group["outcome_1h"] < 0).sum()
    return wins / len(group) if len(group) > 0 else 0

buy_signals = analyzed[analyzed["signal"] == "BUY"]
sell_signals = analyzed[analyzed["signal"] == "SELL"]

print("=== WIN RATE 1h ===")
print(f"BUY:  {win_rate(buy_signals, 'BUY'):.1%} (n={len(buy_signals)})")
print(f"SELL: {win_rate(sell_signals, 'SELL'):.1%} (n={len(sell_signals)})")

print("\n=== RETORNO MÉDIO 1h ===")
print(f"BUY:  {buy_signals['outcome_1h'].mean():.3f}%")
print(f"SELL: {sell_signals['outcome_1h'].mean():.3f}%")

print("\n=== EXPECTANCY (retorno esperado por trade) ===")
buy_expectancy = buy_signals['outcome_1h'].mean()
sell_expectancy = -sell_signals['outcome_1h'].mean()
print(f"BUY expectancy:  {buy_expectancy:.4f}%")
print(f"SELL expectancy: {sell_expectancy:.4f}%")

In [ ]:
# Profit Factor
def profit_factor(group, signal_type):
    if signal_type == "BUY":
        gains = group[group["outcome_1h"] > 0]["outcome_1h"].sum()
        losses = abs(group[group["outcome_1h"] < 0]["outcome_1h"].sum())
    else:
        gains = abs(group[group["outcome_1h"] < 0]["outcome_1h"].sum())
        losses = group[group["outcome_1h"] > 0]["outcome_1h"].sum()
    return gains / losses if losses > 0 else float('inf')

print("=== PROFIT FACTOR ===")
print(f"BUY:  {profit_factor(buy_signals, 'BUY'):.2f}")
print(f"SELL: {profit_factor(sell_signals, 'SELL'):.2f}")

print("\n=== INTERPRETAÇÃO ===")
print("PF < 1.0  → tese morta, pivotar")
print("PF 1.0-1.3 → marginal, filtrar")
print("PF > 1.3  → seguir para Adaptive Sizing")

In [ ]:
# Análise por regime (se disponível)
if "regime" in analyzed.columns and analyzed["regime"].notna().any():
    print("=== WIN RATE POR REGIME ===")
    for regime in analyzed["regime"].dropna().unique():
        subset = analyzed[analyzed["regime"] == regime]
        buy_sub = subset[subset["signal"] == "BUY"]
        if len(buy_sub) > 5:
            wr = (buy_sub["outcome_1h"] > 0).mean()
            print(f"{regime} (BUY, n={len(buy_sub)}): {wr:.1%}")
else:
    print("Coluna 'regime' não populada — verificar core/signals.py")

In [ ]:
# Equity curve simulada (sinais BUY, posição = 1 lote, custo 0.05%)
TRANSACTION_COST_PCT = 0.05  # 0.05% por trade
buy_signals_sorted = buy_signals.sort_values("timestamp_utc").copy()
buy_signals_sorted["pnl_pct"] = buy_signals_sorted["outcome_1h"] - TRANSACTION_COST_PCT
buy_signals_sorted["cumulative_pnl"] = buy_signals_sorted["pnl_pct"].cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(buy_signals_sorted["timestamp_utc"], buy_signals_sorted["cumulative_pnl"])
ax.set_title("Equity Curve Simulada — BUY signals (1h horizon, custo 0.05%)")
ax.set_xlabel("Data")
ax.set_ylabel("PnL Acumulado (%)")
ax.grid(True)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("../knowledge-base/equity_curve_06_06.png", dpi=100)
plt.show()

print(f"PnL final: {buy_signals_sorted['cumulative_pnl'].iloc[-1]:.2f}%")
print(f"Max DD: {(buy_signals_sorted['cumulative_pnl'].cummax() - buy_signals_sorted['cumulative_pnl']).max():.2f}%")

## Decisão Pós-Análise

Preencher após rodar todas as células:

- [ ] **PF BUY:** ___
- [ ] **PF SELL:** ___
- [ ] **Win rate BUY:** ___
- [ ] **Win rate SELL:** ___
- [ ] **Equity curve:** [positiva / lateral / negativa]

**Decisão:**
- [ ] Implementar Adaptive Sizing (PF > 1.3)
- [ ] Filtrar sinais (PF 1.0-1.3)
- [ ] Pivotar tese (PF < 1.0)

**Próximo passo:** _____________